In [0]:
#text widget 

dbutils.widgets.text("event_type_param", "purchase")


In [0]:
#dropdown widget
dbutils.widgets.dropdown(
    "event_type_dropdown",
    "purchase",
    ["purchase", "view", "cart"]
)


In [0]:
#date widget
dbutils.widgets.text("start_date", "2026-01-01")
dbutils.widgets.text("end_date", "2026-01-31")


In [0]:
#read widget value
event_type = dbutils.widgets.get("event_type_dropdown")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
print(event_type)
print(start_date)
print(end_date)

purchase
2026-01-01
2026-01-31


In [0]:
#use widget in silver layer
from pyspark.sql.functions import col

filtered_df = (
    spark.table("ecommerce.events_table")
         .filter(col("event_type") == event_type)
)


In [0]:
#filter by date range
from pyspark.sql.functions import to_date

filtered_df = (
    spark.table("ecommerce.events_table")
         .withColumn("event_date", to_date("event_time"))
         .filter(
             (col("event_date") >= start_date) &
             (col("event_date") <= end_date)
         )
)


In [0]:
from pyspark.sql import functions as F

def process_bronze():
    print("Running Bronze layer...")
    raw_df = spark.read.csv(source_file, header=True, inferSchema=True)
    raw_df.withColumn("ingestion_ts", F.current_timestamp()) \
          .write.format("delta").mode("overwrite").saveAsTable("bronze_events")
    return "Bronze Success"








In [0]:

def process_silver():
    print("Running Silver layer...")
    bronze_df = spark.read.table("bronze_events")

    # Using the cleaning logic from Day 6
    silver_df = bronze_df.filter(F.col("price") > 0) \
                          .dropDuplicates(["user_session", "event_time"]) \
                          .withColumn(
                              "product_name",
                              F.coalesce(
                                  F.element_at(F.split(F.col("category_code"), "\\."), -1),
                                  F.lit("Other")
                              )
                          )

    silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_events")
    return "Silver Success"

In [0]:
def process_gold():
    print("Running Gold layer...")
    silver_df = spark.read.table("silver_events")

    gold_df = silver_df.groupBy("product_id", "product_name") \
                       .agg(F.sum("price").alias("total_revenue"))

    gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_product_revenue")
    return "Gold Success"